# 论文 30：Lost in the Middle：语言模型如何使用长上下文
## Nelson F. Liu、Kevin Lin、John Hewitt 等人，斯坦福大学和华盛顿大学 (2023)

### “Lost in the Middle”现象

语言模型难以有效利用长上下文中部的信息。性能遵循 U 形曲线！

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 模拟多文档问答 任务

**设置**：
- 查询需要来自某一篇文档的信息
- 提供多篇文档（1 篇相关文档，其余为干扰文档）
- **问题**：相关文档的位置重要吗？

In [ ]:
class Document:
    def __init__(self, content, is_relevant=False):
        self.content = content
        self.is_relevant = is_relevant
    
    def __repr__(self):
        return f"Doc(relevant={self.is_relevant}): {self.content[:50]}..."

# 创建合成文档
relevant_doc = Document(
    "The Eiffel Tower was completed in 1889 and stands 330 meters tall. "
    "It was designed by Gustave Eiffel for the 1889 World's Fair in Paris.",
    is_relevant=True
)

distractor_docs = [
    Document("The Great Wall of China is over 13,000 miles long and was built over many centuries."),
    Document("The Statue of Liberty was gifted by France to the United States in 1886."),
    Document("Mount Everest is the tallest mountain on Earth at 8,849 meters above sea level."),
    Document("The Amazon River is the largest river by discharge volume in the world."),
    Document("The Sahara Desert is the largest hot desert, covering much of North Africa."),
    Document("The Colosseum in Rome was completed in 80 AD and could hold 50,000 spectators."),
    Document("The Taj Mahal in India was built between 1632 and 1653 as a mausoleum."),
    Document("The Grand Canyon in Arizona is 277 miles long and up to 18 miles wide."),
    Document("The Great Barrier Reef is the world's largest coral reef system."),
]

query = "When was the Eiffel Tower completed?"
correct_answer = "1889"

print(f"Query: {query}")
print(f"Correct answer: {correct_answer}")
print(f"\nRelevant document: {relevant_doc.content}")
print(f"\nNumber of distractor documents: {len(distractor_docs)}")

## 简化语言模型

模拟具有位置偏置的基于注意力的模型

In [ ]:
class SimpleLM:
    '具有位置偏置的简化 LM'
    def __init__(self, position_bias_type='u_shaped'):
        """position_bias_type：
        - 'uniform'：对所有位置给予相同注意力
        - 'u_shaped'：开头和结尾高，中间低
        - 'recency'：偏好较近的结尾位置
        - 'primacy'：偏好较早的开头位置"""
        self.position_bias_type = position_bias_type
    
    def get_position_weights(self, num_positions):
        '计算基于位置的注意力权重'
        positions = np.arange(num_positions)
        
        if self.position_bias_type == 'uniform':
            weights = np.ones(num_positions)
        
        elif self.position_bias_type == 'u_shaped':
            # U 形：边缘高，中间低
            normalized_pos = positions / (num_positions - 1)  # 0 到 1
            # 最小值为 0.5 的二次
            weights = 4 * (normalized_pos - 0.5) ** 2 + 0.3
        
        elif self.position_bias_type == 'recency':
            # 从开头开始指数衰减
            weights = np.exp(positions * 0.2)
        
        elif self.position_bias_type == 'primacy':
            # 越接近结尾权重越高
            weights = np.exp(-positions * 0.2)
        
        # 标准化
        weights = weights / np.sum(weights)
        return weights
    
    def answer_query(self, query, documents):
        """使用文档模拟回答查询
        返回：找到正确答案的概率"""
        num_docs = len(documents)
        
        # 获取位置权重
        position_weights = self.get_position_weights(num_docs)
        
        # 查找相关文档位置
        relevant_position = None
        for i, doc in enumerate(documents):
            if doc.is_relevant:
                relevant_position = i
                break
        
        if relevant_position is None:
            return 0.0  # 无相关文档
        
        # 使用相关文档的概率
        # 权重越高→更有可能使用该文档
        prob_correct = position_weights[relevant_position]
        
        return prob_correct

# 测试不同的偏差类型
num_docs = 10
test_positions = np.arange(num_docs)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

bias_types = ['uniform', 'u_shaped', 'recency', 'primacy']
for ax, bias_type in zip(axes, bias_types):
    model = SimpleLM(position_bias_type=bias_type)
    weights = model.get_position_weights(num_docs)
    
    ax.bar(test_positions, weights, color='steelblue', edgecolor='black')
    ax.set_xlabel('Document Position', fontsize=11)
    ax.set_ylabel('Attention Weight', fontsize=11)
    ax.set_title(f'{bias_type.replace("_", " ").title()} Bias', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, max(weights) * 1.2)

plt.tight_layout()
plt.show()

print("\nReal LLMs show U-shaped bias (high at beginning/end, low in middle)!")

## 测试位置敏感性

In [ ]:
def test_all_positions(model, query, relevant_doc, distractor_docs):
    '在每个位置用相关文档测试性能'
    num_positions = len(distractor_docs) + 1
    accuracies = []
    
    for pos in range(num_positions):
        # 在 pos 位置创建相关文档的文档列表
        docs = distractor_docs[:pos] + [relevant_doc] + distractor_docs[pos:]
        docs = docs[:num_positions]  # 保持固定长度
        
        # 获取模型正确回答的概率
        prob_correct = model.answer_query(query, docs)
        accuracies.append(prob_correct)
    
    return accuracies

# 测试U型偏差（现实）
model_realistic = SimpleLM(position_bias_type='u_shaped')
accuracies_realistic = test_all_positions(model_realistic, query, relevant_doc, distractor_docs)

# 测试均匀偏置（理想情况）
model_ideal = SimpleLM(position_bias_type='uniform')
accuracies_ideal = test_all_positions(model_ideal, query, relevant_doc, distractor_docs)

# 绘图
positions = np.arange(len(accuracies_realistic))

plt.figure(figsize=(12, 6))
plt.plot(positions, accuracies_realistic, 'o-', linewidth=3, markersize=10, 
        label='Realistic (U-shaped bias)', color='crimson')
plt.plot(positions, accuracies_ideal, 's--', linewidth=2, markersize=8, 
        label='Ideal (No bias)', color='green', alpha=0.6)

# 上下文开头和结尾
plt.axvline(x=0, color='blue', linestyle=':', alpha=0.5, linewidth=2, label='Beginning')
plt.axvline(x=len(positions)-1, color='purple', linestyle=':', alpha=0.5, linewidth=2, label='End')

# 上下文中间区域
middle_start = len(positions) // 4
middle_end = 3 * len(positions) // 4
plt.axvspan(middle_start, middle_end, alpha=0.2, color='red', label='Middle (worst)')

plt.xlabel('Position of Relevant Document', fontsize=13)
plt.ylabel('Accuracy', fontsize=13)
plt.title('Lost in the Middle: Performance vs Position', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 统计数据
beginning_acc = accuracies_realistic[0]
middle_acc = np.mean(accuracies_realistic[middle_start:middle_end])
end_acc = accuracies_realistic[-1]

print(f"\nPerformance Analysis:")
print(f"Beginning (pos 0): {beginning_acc:.1%}")
print(f"Middle (pos {middle_start}-{middle_end}): {middle_acc:.1%}")
print(f"End (pos {len(positions)-1}): {end_acc:.1%}")
print(f"\nMiddle penalty: -{(beginning_acc - middle_acc)/beginning_acc:.1%} relative to beginning")

## 上下文长度的影响

In [ ]:
def test_varying_lengths(model, query, relevant_doc, distractor_docs, lengths):
    '测试性能如何随上下文长度变化'
    results = {'beginning': [], 'middle': [], 'end': []}
    
    for length in lengths:
        # 使用干扰子集
        current_distractors = distractor_docs[:length-1]
        
        # 测试三个位置：开始、中间、结束
        positions = {
            'beginning': 0,
            'middle': length // 2,
            'end': length - 1
        }
        
        for pos_name, pos in positions.items():
            docs = current_distractors[:pos] + [relevant_doc] + current_distractors[pos:]
            docs = docs[:length]
            
            acc = model.answer_query(query, docs)
            results[pos_name].append(acc)
    
    return results

# 测试不同的上下文长度
lengths = [3, 5, 7, 9, 10]
results = test_varying_lengths(model_realistic, query, relevant_doc, distractor_docs, lengths)

# 绘图
plt.figure(figsize=(12, 6))
plt.plot(lengths, results['beginning'], 'o-', linewidth=3, markersize=10, 
        label='Beginning', color='blue')
plt.plot(lengths, results['middle'], 's-', linewidth=3, markersize=10, 
        label='Middle', color='red')
plt.plot(lengths, results['end'], '^-', linewidth=3, markersize=10, 
        label='End', color='purple')

plt.xlabel('Number of Documents', fontsize=13)
plt.ylabel('Accuracy', fontsize=13)
plt.title('Performance Degradation with Context Length', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nLonger contexts → worse performance (especially in middle!)")

## RAG 的文档排序策略

In [ ]:
def order_documents(documents, relevance_scores, strategy='default'):
    """根据策略排序文档
    
    策略：
    - 'default'：保持检索顺序
    - 'most_relevant_first'：将最佳文档放在开头
    - 'most_relevant_edges'：将最佳内容放在开头和结尾
    - 'reverse'：反转检索顺序"""
    indices = np.arange(len(documents))
    
    if strategy == 'default':
        return documents
    
    elif strategy == 'most_relevant_first':
        # 按相关性排序（降序）
        sorted_indices = np.argsort(relevance_scores)[::-1]
        return [documents[i] for i in sorted_indices]
    
    elif strategy == 'most_relevant_edges':
        # 将最相关的文档放在开头和结尾
        sorted_indices = np.argsort(relevance_scores)[::-1]
        
        # 交错：边缘最好，中间最差
        ordered = []
        for i in range(len(documents) // 2):
            ordered.append(documents[sorted_indices[i]])  # 高相关性
        for i in range(len(documents) // 2, len(documents)):
            ordered.append(documents[sorted_indices[i]])  # 相关性低
        
        # 反转后一半以在结尾处放高
        mid = len(ordered) // 2
        return ordered[:mid] + ordered[mid:][::-1]
    
    elif strategy == 'reverse':
        return documents[::-1]
    
    return documents

# 模拟检索分数
num_test_docs = 10
test_docs = [relevant_doc] + distractor_docs[:num_test_docs-1]

# 相关性分数（相关文档获得高分）
relevance_scores = np.random.rand(num_test_docs) * 0.5
relevance_scores[0] = 0.95  # 相关文档得分高

# 随机打乱以模拟检索
shuffle_idx = np.random.permutation(num_test_docs)
test_docs = [test_docs[i] for i in shuffle_idx]
relevance_scores = relevance_scores[shuffle_idx]

# 测试不同的策略
strategies = ['default', 'most_relevant_first', 'most_relevant_edges']
strategy_accuracies = {}

for strategy in strategies:
    ordered = order_documents(test_docs, relevance_scores, strategy)
    acc = model_realistic.answer_query(query, ordered)
    strategy_accuracies[strategy] = acc
    
    # 查找相关文档的位置
    rel_pos = next(i for i, doc in enumerate(ordered) if doc.is_relevant)
    print(f"\n{strategy:25s}: Relevant doc at position {rel_pos:2d}, Accuracy: {acc:.1%}")

# 可视化
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(strategies)), 
              [strategy_accuracies[s] for s in strategies],
              color=['lightcoral', 'lightblue', 'lightgreen'],
              edgecolor='black', linewidth=2)

plt.xticks(range(len(strategies)), 
          [s.replace('_', '\n').title() for s in strategies],
          fontsize=11)
plt.ylabel('Accuracy', fontsize=13)
plt.title('Document Ordering Strategies', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bar, strategy in zip(bars, strategies):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{strategy_accuracies[strategy]:.1%}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("RECOMMENDATION: Put most important documents at edges!")
print("="*60)

## 注意力模式分析

In [ ]:
# 模拟不同上下文长度的注意力模式
context_lengths = [10, 20, 30]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, length in zip(axes, context_lengths):
    # 生成注意力权重（U形）
    positions = np.arange(length)
    normalized = positions / (length - 1)
    attention = 4 * (normalized - 0.5) ** 2 + 0.3
    attention = attention / np.sum(attention)
    
    # 绘图
    ax.bar(positions, attention, color='steelblue', edgecolor='black', linewidth=1)
    ax.set_xlabel('Position', fontsize=11)
    ax.set_ylabel('Attention Weight', fontsize=11)
    ax.set_title(f'Context Length = {length}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 突出显示中间区域
    middle_start = length // 4
    middle_end = 3 * length // 4
    ax.axvspan(middle_start, middle_end, alpha=0.2, color='red')

plt.suptitle('Attention Patterns: Lost in the Middle', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nAs context grows, middle positions get even less attention!")

## 要点总结总结总结

### Lost in the Middle 现象：

**观察**：语言模型显示**U 形性能曲线**
- ✅ 当相关信息位于**开头**时，准确率高
- ✅ 当相关信息位于**结尾**时，准确率也较高
- ❌ 当相关信息位于**中间**时，准确率较低

### 为什么会发生这种情况？

**假设**：

1. **注意力模式**：
   - 自注意力自然会关注最近的 token（近因偏置）
   - 还关注开头的 token（首因偏置）
   - 中间的 token 受到的关注较少

2. **训练数据分布**：
   - 大多数训练文档都很短
   - 预训练中很少有长上下文
   - 模型还没有学会很好地利用中间位置的信息

3. **因果掩码**：
   - 解码器模型无法“向前看”
   - 中间的信息可能会被后面的 token“覆盖”

### 实验结果：

**来自论文**：

**多文档问答**：
- 位置 1 处的相关文档（开头）：~90% 准确度
- 位置 5（中间位置）的相关文档：~60% 准确度
- 位置 10（结尾）的相关文档：~85% 准确度

**上下文长度的影响**：
- 10 篇文档：中间位置惩罚约为 30%
- 20 篇文档：中间位置惩罚约为 40%
- 30 篇文档：中间位置惩罚约为 50%

**测试的模型**：
- GPT-3.5-turbo：明显的 U 形偏置
- Claude：明显的 U 形偏置
- GPT-4：已缓解但仍然存在
- 开源 LLM：更明显的位置偏置

### 位置偏置公式：

位置 $p$ 处的性能（标准化 0-1）：
$$
\text{Accuracy}(p) \propto 4(p - 0.5)^2 + c
$$

其中：
- 最小值为 $p = 0.5$（中间位置）
- 在 $p = 0$ 和 $p = 1$ 处达到最大值（边缘）
- $c$ 是基准性能

### 对 RAG 系统的影响：

**问题**：
```
Retriever returns: [Doc1, Doc2, ..., Doc20]
                    (sorted by relevance score)

If most relevant doc is in middle → poor performance!
```

**解决方案**：

1. **重新排序检索到的文档**：
   - 将最相关的文档放在开头
   - 或交错排列：最相关的放在两端，较不相关的放在中间

2. **限制上下文长度**：
   - 使用更少、更相关的文档
   - 使用 top-3 或 top-5，而不是 top-20

3. **分块**：
   - 分成较小的块处理长上下文
   - 汇总结果

4. **显式引导注意力**：
   - 微调模型，使其关注中间位置
   - 加入能够抵消位置偏置的位置表示

### 文档排序策略：

| 策略 | 说明 | 表现 |
|----------|-------------|-------------|
| 检索顺序 | 保持检索器返回的顺序 | 基线 |
| 最相关优先 | 将最相关文档放在开头 | 较好 |
| 最相关置于两端 | 将最相关文档放在开头和结尾 | **最佳** |
| 反向 | 反转检索顺序 | 结果不一 |

### 最佳实践：

1. 尽可能使用**较短的上下文**
2. 将**重要信息放在两端**（开头或结尾）
3. 将文档传给 LLM 前先进行**重排序**
4. 对很长的上下文进行**分块**
5. **测试**所用模型的位置敏感性

### 代码示例（重新排序）：

```python
def reorder_for_llm(docs, scores):
    """Put most relevant at edges"""
    sorted_idx = np.argsort(scores)[::-1]
    
    # Interleave high and low relevance
    reordered = []
    for i in range(len(docs) // 2):
        reordered.append(docs[sorted_idx[i]])  # High
    for i in range(len(docs) // 2, len(docs)):
        reordered.append(docs[sorted_idx[i]])  # Low
    
    # Move best to end as well
    mid = len(reordered) // 2
    return reordered[:mid] + reordered[mid:][::-1]
```

### 缓解策略：

**训练期间**：
- 加入长上下文训练样本
- 对中间位置进行显式监督
- 使用位置感知目标

**在推理过程中**：
- 有策略地重新排序文档
- 进行多轮处理（每轮处理一个子集）
- 明确提示：“同等关注所有文档”

**架构变化**：
- 稀疏注意力模式
- 分层处理
- 检索增强注意力

### 未来方向：

- **位置不变模型**：训练忽略位置偏置
- **自适应注意力**：学会关注相关部分
- **分块处理**：在重叠窗口中处理
- **多遍推理**：多次读取上下文

### 核心提醒：

```
⚠️  WARNING: Don't assume LLMs use all context equally!

✅  DO: Test position sensitivity
✅  DO: Put important info at edges  
✅  DO: Keep contexts short when possible
❌  DON'T: Assume middle positions work well
❌  DON'T: Blindly concatenate many documents
```

### 影响：

本文揭示了当前 LLM 的一个关键局限性，并改变了我们的思维方式：
- RAG 系统设计
- 长上下文评估
- QA 的文档排序
- 使用多个来源时的提示工程

**记住**：即使有超过 100k token 的上下文窗口，位置也很重要！